In [38]:
import pandas as pd
import numpy as np
import requests
import bs4 as bs
import urllib.request

In [39]:
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2020"

source = urllib.request.urlopen(link).read()
soup = bs.BeautifulSoup(source,'lxml')


In [40]:
tables= soup.find_all('table',class_='wikitable sortable')


In [41]:
df1=pd.read_html(str(tables[0]))[0]
df2=pd.read_html(str(tables[1]))[0]
df3=pd.read_html(str(tables[2]))[0]
df4=pd.read_html(str(tables[3]).replace("'1\"\'",'"1"'))[0]


C:\Users\Vishnu\AppData\Local\Temp\ipykernel_26720\559264434.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df1=pd.read_html(str(tables[0]))[0]
C:\Users\Vishnu\AppData\Local\Temp\ipykernel_26720\559264434.py:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df2=pd.read_html(str(tables[1]))[0]
C:\Users\Vishnu\AppData\Local\Temp\ipykernel_26720\559264434.py:3: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df3=pd.read_html(str(tables[2]))[0]
C:\Users\Vishnu\AppData\Local\Temp\ipykernel_26720\559264434.py:4: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To re

In [42]:
df=pd.concat([df1,df2,df3,df4],ignore_index=True)

In [43]:
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,3,The Grudge,Screen Gems / Stage 6 Films / Ghost House Pict...,Nicolas Pesce (director/screenplay); Andrea Ri...,[2]
1,J A N U A R Y,10,Underwater,20th Century Fox / TSG Entertainment / Chernin...,"William Eubank (director); Brian Duffield, Ada...",[3]
2,J A N U A R Y,10,Like a Boss,Paramount Pictures / Artists First,"Miguel Arteta (director); Sam Pitman, Adam Col...",[4]
3,J A N U A R Y,10,Three Christs,IFC Films,Jon Avnet (director/screenplay); Eric Nazarian...,NaN
4,J A N U A R Y,10,Inherit the Viper,Lionsgate / Barry Films / Tycor International ...,Anthony Jerjen (director); Andrew Crabtree (sc...,[5]
...,...,...,...,...,...,...
272,D E C E M B E R,25,We Can Be Heroes,Netflix / Troublemaker Studios / Double R Prod...,Robert Rodriguez (director/screenplay); Priyan...,[245]
273,D E C E M B E R,25,News of the World,Universal Pictures / Playtone / Perfect World ...,Paul Greengrass (director/screenplay); Luke Da...,[246]
274,D E C E M B E R,25,One Night in Miami...,Amazon Studios,Regina King (director); Kemp Powers (screenpla...,[247]
275,D E C E M B E R,25,Promising Young Woman,Focus Features / FilmNation Entertainment,Emerald Fennell (director/screenplay); Carey M...,[248]


In [44]:
df_2020 =df[["Title",'Cast and crew']]

In [45]:
df_2020.head()

,Title,Cast and crew
0,The Grudge,Nicolas Pesce (director/screenplay); Andrea Ri...
1,Underwater,"William Eubank (director); Brian Duffield, Ada..."
2,Like a Boss,"Miguel Arteta (director); Sam Pitman, Adam Col..."
3,Three Christs,Jon Avnet (director/screenplay); Eric Nazarian...
4,Inherit the Viper,Anthony Jerjen (director); Andrew Crabtree (sc...


In [46]:
import json

In [55]:
print(df_2020.columns)


Index(['Title', 'Cast and crew'], dtype='object')


In [56]:
df_2020['Title'] = df_2020['Title'].astype(str)  # Convert to string


C:\Users\Vishnu\AppData\Local\Temp\ipykernel_26720\1423428002.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2020['Title'] = df_2020['Title'].astype(str)  # Convert to string


In [57]:
from tmdbv3api import TMDb
import json
import requests
tmdb =TMDb()
tmdb.api_key ='08d38c60a56fe3fdf55456105dab7193'

In [64]:
import numpy as np
import requests
from tmdbv3api import Movie

tmdb_movie = Movie()

def get_genre(x):
    try:
        if not isinstance(x, str) or not x.strip():  # Ignore empty or invalid titles
            return np.NaN

        print(f"Processing title: {x}")  # Debugging output
        genres = []
        result = tmdb_movie.search(x)

        if not result:
            print(f"No results found for: {x}")
            return np.NaN

        movie_id = result[0].id
        url = f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}'
        response = requests.get(url)
        data_json = response.json()

        if "genres" in data_json and data_json["genres"]:
            genres = [g['name'] for g in data_json["genres"]]
            return " | ".join(genres)

        return np.NaN

    except Exception as e:
        print(f"Error processing '{x}': {e}")
        return np.NaN  # Return NaN for any error to avoid crashes


In [67]:
df_2020 = df_2020[df_2020['Title'].notna()]  # Remove NaN values
df_2020['Title'] = df_2020['Title'].astype(str)  # Convert all titles to string

df_2020['genres'] = df_2020['Title'].map(get_genre)  # Apply function


Processing title: The Grudge
Processing title: Underwater
Processing title: Like a Boss
Processing title: Three Christs
Processing title: Inherit the Viper
Processing title: The Sonata
Processing title: The Murder of Nicole Brown Simpson
Processing title: Angels Fallen
Processing title: Bad Boys for Life
Processing title: Dolittle
Processing title: A Fall from Grace
Processing title: The Gentlemen
Processing title: The Turning
Processing title: The Last Full Measure
Processing title: John Henry
Processing title: The Rhythm Section
Processing title: Gretel & Hansel
Processing title: The Assistant
Processing title: Birds of Prey
Processing title: The Lodge
Processing title: Timmy Failure: Mistakes Were Made
Processing title: Horse Girl
Processing title: To All the Boys: P.S. I Still Love You
Processing title: Sonic the Hedgehog
Processing title: Fantasy Island
Processing title: The Photograph
Processing title: Downhill
Processing title: Spy Intervention
Processing title: The Kindness of 

In [1]:
from tmdbv3api import TMDb

tmdb = TMDb()
tmdb.api_key = '08d38c60a56fe3fdf55456105dab7193'  # Replace with your key

# Test if the key is valid
try:
    from tmdbv3api import Movie
    movie = Movie()
    test_movie = movie.details(550)  # Movie ID 550 = Fight Club
    print("✅ API Key is valid! TMDb is working.")
except Exception as e:
    print("❌ Invalid API Key or connection issue:", e)


✅ API Key is valid! TMDb is working.


In [68]:
df_2020.head()

,Title,Cast and crew,genres
0,The Grudge,Nicolas Pesce (director/screenplay); Andrea Ri...,Horror | Mystery | Thriller
1,Underwater,"William Eubank (director); Brian Duffield, Ada...",Horror | Science Fiction | Action | Adventure
2,Like a Boss,"Miguel Arteta (director); Sam Pitman, Adam Col...",Comedy
3,Three Christs,Jon Avnet (director/screenplay); Eric Nazarian...,Drama
4,Inherit the Viper,Anthony Jerjen (director); Andrew Crabtree (sc...,Crime | Thriller | Drama


In [69]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [70]:
df_2020['director_name']= df_2020['Cast and crew'].map(lambda x: get_director(x))


In [71]:
def get_actor1(x):
    return ((x.split("screenplay); ")[-1]).split(', ')[0])

In [72]:
df_2020["actor_1_name"]= df_2020["Cast and crew"].map(lambda x: get_actor1(x))

In [73]:
def get_actor2(x):
    if len((x.split("screenplay); ")[-1]).split(', '))<2:
        return np.NaN
    else:
        return ((x.split("screenplay); ")[-1]).split(', ')[1])

In [74]:
df_2020["actor_2_name"]=df_2020["Cast and crew"].map(lambda x: get_actor2(x))

In [76]:
def get_actor3(x):
    if len((x.split("screenplay); ")[-1]).split(', '))<3:
        return np.NaN
    else:
        return ((x.split("screenplay); "))[-1].split(', ')[2])

In [77]:
df_2020["actor_3_name"]=df_2020["Cast and crew"].map(lambda x: get_actor3(x))

In [78]:
df_2020

,Title,Cast and crew,genres,director_name,actor_1_name,actor_2_name,actor_3_name
0,The Grudge,Nicolas Pesce (director/screenplay); Andrea Ri...,Horror | Mystery | Thriller,Nicolas Pesce,Andrea Riseborough,Demián Bichir,John Cho
1,Underwater,"William Eubank (director); Brian Duffield, Ada...",Horror | Science Fiction | Action | Adventure,William Eubank,Kristen Stewart,Vincent Cassel,Jessica Henwick
2,Like a Boss,"Miguel Arteta (director); Sam Pitman, Adam Col...",Comedy,Miguel Arteta,Tiffany Haddish,Rose Byrne,Salma Hayek
3,Three Christs,Jon Avnet (director/screenplay); Eric Nazarian...,Drama,Jon Avnet,Richard Gere,Peter Dinklage,Walton Goggins
4,Inherit the Viper,Anthony Jerjen (director); Andrew Crabtree (sc...,Crime | Thriller | Drama,Anthony Jerjen,Josh Hartnett,Margarita Levieva,Chandler Riggs
...,...,...,...,...,...,...,...
272,We Can Be Heroes,Robert Rodriguez (director/screenplay); Priyan...,Family | Action | Fantasy | Comedy,Robert Rodriguez,Priyanka Chopra Jonas,Pedro Pascal,YaYa Gosselin
273,News of the World,Paul Greengrass (director/screenplay); Luke Da...,Drama | Western | Adventure,Paul Greengrass,Tom Hanks,Helena Zengel,NaN
274,One Night in Miami...,Regina King (director); Kemp Powers (screenpla...,Drama,Regina King,Kingsley Ben-Adir,Eli Goree,Aldis Hodge
275,Promising Young Woman,Emerald Fennell (director/screenplay); Carey M...,Thriller | Crime | Drama,Emerald Fennell,Carey Mulligan,Bo Burnham,Alison Brie


In [79]:
df_2020['genres'] = df_2020['genres'].str.replace(r'\s*\|\s*', ' ', regex=True)


In [80]:
df_2020.head()

,Title,Cast and crew,genres,director_name,actor_1_name,actor_2_name,actor_3_name
0,The Grudge,Nicolas Pesce (director/screenplay); Andrea Ri...,Horror Mystery Thriller,Nicolas Pesce,Andrea Riseborough,Demián Bichir,John Cho
1,Underwater,"William Eubank (director); Brian Duffield, Ada...",Horror Science Fiction Action Adventure,William Eubank,Kristen Stewart,Vincent Cassel,Jessica Henwick
2,Like a Boss,"Miguel Arteta (director); Sam Pitman, Adam Col...",Comedy,Miguel Arteta,Tiffany Haddish,Rose Byrne,Salma Hayek
3,Three Christs,Jon Avnet (director/screenplay); Eric Nazarian...,Drama,Jon Avnet,Richard Gere,Peter Dinklage,Walton Goggins
4,Inherit the Viper,Anthony Jerjen (director); Andrew Crabtree (sc...,Crime Thriller Drama,Anthony Jerjen,Josh Hartnett,Margarita Levieva,Chandler Riggs


In [81]:
df_2020 = df_2020.rename(columns={'Title':"movie_title"})

In [82]:
new_df20 = df_2020.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

In [83]:
new_df20.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title
0,Nicolas Pesce,Andrea Riseborough,Demián Bichir,John Cho,Horror Mystery Thriller,The Grudge
1,William Eubank,Kristen Stewart,Vincent Cassel,Jessica Henwick,Horror Science Fiction Action Adventure,Underwater
2,Miguel Arteta,Tiffany Haddish,Rose Byrne,Salma Hayek,Comedy,Like a Boss
3,Jon Avnet,Richard Gere,Peter Dinklage,Walton Goggins,Drama,Three Christs
4,Anthony Jerjen,Josh Hartnett,Margarita Levieva,Chandler Riggs,Crime Thriller Drama,Inherit the Viper


In [84]:
new_df20['comb']= new_df20['actor_1_name']+ ' '+ new_df20['actor_2_name']+ " "+ new_df20['actor_3_name']+ " "+ new_df20['director_name']+ " "+ new_df20['genres']

In [85]:
new_df20.isna().sum()

director_name     0
actor_1_name      0
actor_2_name      4
actor_3_name     27
genres            1
movie_title       0
comb             28
dtype: int64

In [86]:
new_df20['movie_title'] = new_df20['movie_title'].str.lower()

In [87]:
new_df20.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Nicolas Pesce,Andrea Riseborough,Demián Bichir,John Cho,Horror Mystery Thriller,the grudge,Andrea Riseborough Demián Bichir John Cho Nico...
1,William Eubank,Kristen Stewart,Vincent Cassel,Jessica Henwick,Horror Science Fiction Action Adventure,underwater,Kristen Stewart Vincent Cassel Jessica Henwick...
2,Miguel Arteta,Tiffany Haddish,Rose Byrne,Salma Hayek,Comedy,like a boss,Tiffany Haddish Rose Byrne Salma Hayek Miguel ...
3,Jon Avnet,Richard Gere,Peter Dinklage,Walton Goggins,Drama,three christs,Richard Gere Peter Dinklage Walton Goggins Jon...
4,Anthony Jerjen,Josh Hartnett,Margarita Levieva,Chandler Riggs,Crime Thriller Drama,inherit the viper,Josh Hartnett Margarita Levieva Chandler Riggs...


In [88]:
new_df20=new_df20.dropna(how='any')

In [89]:
new_df20.isna().sum()

director_name    0
actor_1_name     0
actor_2_name     0
actor_3_name     0
genres           0
movie_title      0
comb             0
dtype: int64

In [91]:
old_df = pd.read_csv('../datasets/final_data.csv')
old_df.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...


In [92]:
final_df =pd.concat([old_df,new_df20],ignore_index=True)

In [93]:
final_df.head()

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...


In [94]:
final_df.to_csv('../datasets/main_data.csv',index=False)